## Week 4.2: Modules and packages.

A notebook based on the [Python tutorial](https://docs.python.org/3/tutorial), which you are invited to consult.

#### Table of contents:
**The notebook is organized as follows:**

* **[Part 1: Python scopes and namespace](#421-python-scopes-and-namespace)**

* **[Part 2: Modules](#422-modules)**

* **[Part 3: Packages](#423-packages)**


### 4.2.1 Python scopes and namespace

A **namespace** is a mapping from names to objects. Examples of namespaces include:

- the set of built-in names (functions such as `abs()`, and built-in exception names)
- the global names in a module
- the local names in a function invocation

Namespaces are created at different moments and have different lifetimes. Different namespaces can contain the same name without any collision — for instance, both module `A` and module `B` could each define a function `maxvalue` without confusion, as long as you prefix it with the module name when using it from outside (`A.maxvalue`, `B.maxvalue`).

An **attribute** is any name following a dot — e.g. in `z.real`, `real` is an attribute of the object `z`.

A **scope** is a textual region of a Python program where a namespace is directly accessible (i.e., where an unqualified reference to a name attempts to find it). At any time during execution there are (at least) three or four nested scopes whose namespaces are directly accessible, searched in this order:

1. the innermost scope, containing local names
2. the scopes of any enclosing functions, containing non-local, non-global names
3. the next-to-last scope, containing the current module's global names
4. the outermost scope, containing built-in names

This search order is often called the **LEGB rule**: **L**ocal, **E**nclosing, **G**lobal, **B**uilt-in.

In [ ]:
def scope_test():
    def do_local():
        spam = "local spam"

    def do_nonlocal():
        nonlocal spam
        spam = "nonlocal spam"

    def do_global():
        global spam
        spam = "global spam"

    spam = "test spam"
    do_local()
    print("After local assignment:", spam)
    do_nonlocal()
    print("After nonlocal assignment:", spam)
    do_global()
    print("After global assignment:", spam)

scope_test()
print("In global scope:", spam)

Notice how the *local* assignment (which is default) didn't change `scope_test`'s binding of `spam`. The `nonlocal` assignment changed `scope_test`'s binding of `spam`, and the `global` assignment changed the module-level binding.

You can also see that there was no previous binding for `spam` before the `global` assignment — it was created in the module's global namespace by that assignment.

#### The `global` statement

The `global` statement is used to indicate that particular variables live in the global (module) scope and should be rebound there.

In [ ]:
counter = 0

def increment():
    global counter
    counter += 1

increment()
increment()
increment()
print(counter)

#### The `nonlocal` statement

The `nonlocal` statement is used inside nested functions to indicate that a particular variable lives in an enclosing scope (but not the global scope) and should be rebound there. This is the classic way to build a *closure*.

In [ ]:
def make_counter():
    count = 0
    def counter():
        nonlocal count
        count += 1
        return count
    return counter

c = make_counter()
print(c())
print(c())
print(c())

#### Exercise 4.2.1

Predict what the following code prints, *then* run the cell to check your answer.

In [ ]:
x = "global"

def outer():
    x = "enclosing"

    def inner():
        x = "local"
        print("inner:", x)

    inner()
    print("outer:", x)

outer()
print("module:", x)

#### `dir()` and namespaces

The built-in function `dir()` is used to find out which names a module (or the current scope) defines. Without arguments, `dir()` lists the names you have defined in the current scope.

In [ ]:
a_variable = 42
def a_function():
    pass

names = dir()
[n for n in names if not n.startswith('__')]

Note that `dir()` does **not** list the names of built-in functions and variables by default. If you want that list, they are defined in the standard module `builtins`:

In [ ]:
import builtins
len(dir(builtins))

### 4.2.2 Modules

As your programs get longer, you'll want to split them into several files for easier maintenance. A **module** is a file containing Python definitions and statements. The file name is the module name with the suffix `.py` appended.

Within a module, the module's name (as a string) is available as the value of the global variable `__name__`.

For example, suppose you have a file `fibo.py` in the current directory with the following contents:

```python
# Fibonacci numbers module

def fib(n):    # write Fibonacci series up to n
    a, b = 0, 1
    while a < n:
        print(a, end=' ')
        a, b = b, a+b
    print()

def fib2(n):   # return Fibonacci series up to n
    result = []
    a, b = 0, 1
    while a < n:
        result.append(a)
        a, b = b, a+b
    return result
```

You would then enter the Python interpreter and import this module with `import fibo`. This does not enter the names of the functions defined in `fibo` directly in the current namespace — it only enters the module name `fibo` there. Using the module name you can access the functions: `fibo.fib(1000)`.

Let's create this module on disk and explore it for real.

In [ ]:
fibo_source = '''
# Fibonacci numbers module

def fib(n):    # write Fibonacci series up to n
    a, b = 0, 1
    while a < n:
        print(a, end=' ')
        a, b = b, a + b
    print()

def fib2(n):   # return Fibonacci series up to n
    result = []
    a, b = 0, 1
    while a < n:
        result.append(a)
        a, b = b, a + b
    return result
'''

with open('fibo.py', 'w') as f:
    f.write(fibo_source)

print('fibo.py written to disk')

In [ ]:
import fibo

fibo.fib(1000)

In [ ]:
fibo.fib2(100)

In [ ]:
fibo.__name__

If you intend to use a function often, you can assign it to a local name:

In [ ]:
fib = fibo.fib
fib(500)

#### More on modules

A module can contain executable statements as well as function definitions. These statements are intended to initialize the module and are executed only the **first time** the module name is encountered in an import statement (they are also run if the file is executed as a script).

Each module has its own private namespace, used as the global namespace by all functions defined in the module. This means the author of a module can use global variables without worrying about accidental clashes with a user's global variables.

Modules can import other modules. It is customary (but not required) to place all `import` statements at the beginning of a module.

There is a variant of the `import` statement that imports names from a module directly into the importing module's namespace:

In [ ]:
from fibo import fib, fib2
fib(200)

There is even a variant to import all names that a module defines, using `from fibo import *`. This imports all names except those beginning with an underscore. **In general, this practice is discouraged** since it introduces an unknown set of names into the interpreter, possibly hiding some things you have already defined, and makes it much harder for readers to know where a name came from.

If the module name is followed by `as`, then the name following `as` is bound directly to the imported module:

In [ ]:
import fibo as fib_module
fib_module.fib(30)

In [ ]:
from fibo import fib as fibonacci
fibonacci(50)

#### Executing modules as scripts

When you run a Python module with:

```bash
python fibo.py <arguments>
```

the code in the module will be executed, just as if you imported it, **but with `__name__` set to `"__main__"`**. By adding this code at the end of your module:

```python
if __name__ == "__main__":
    import sys
    fib(int(sys.argv[1]))
```

you can make the file usable both as a script and as an importable module. This is an extremely common idiom — it's a way to write code that only runs when the file is executed directly, not when it's imported elsewhere.

In [ ]:
script_addition = '''

if __name__ == "__main__":
    import sys
    fib(int(sys.argv[1]) if len(sys.argv) > 1 else 100)
'''

with open('fibo.py', 'a') as f:
    f.write(script_addition)

print('Updated fibo.py:')
with open('fibo.py') as f:
    print(f.read())

In [ ]:
import subprocess
result = subprocess.run(['python3', 'fibo.py', '300'], capture_output=True, text=True)
print(result.stdout)

#### The Module Search Path

When a module named `spam` is imported, the interpreter first searches for a built-in module with that name. If not found, it searches for a file named `spam.py` in a list of directories given by the variable `sys.path`. `sys.path` is initialized from these locations:

- the directory containing the input script (or the current directory when no file is specified)
- `PYTHONPATH` (a list of directory names, with the same syntax as the shell variable `PATH`)
- the installation-dependent default (including the `site-packages` directory, handled by the `site` module)

In [ ]:
import sys
sys.path[:5]  # first few entries of the module search path

#### "Compiled" Python files

To speed up loading modules, Python caches the compiled version of each module in the `__pycache__` directory under the name `module.version.pyc`. Python checks the modification date of the source against the compiled version to see whether it's out of date and needs to be recompiled. This is entirely automatic.

In [ ]:
fib(50)  # trigger a re-import indirectly by importing again in a fresh way
import importlib
importlib.reload(fibo)
import os
os.listdir('.')

#### Standard modules

Python comes with a library of standard modules, described in the Python Library Reference. Some modules are built into the interpreter; these provide access to operations that are not part of the core of the language but are nevertheless built in, either for efficiency or to provide access to operating system primitives such as system calls. One example is the `sys` module.

In [ ]:
import sys
print("Python version:", sys.version.split()[0])
print("Platform:", sys.platform)

#### The `dir()` function applied to modules

The built-in function `dir()` is used to find out which names a module defines. It returns a sorted list of strings.

In [ ]:
dir(fibo)

Without arguments, `dir()` lists the names you have defined *currently* — recall from Part 1 that this reflects your current namespace, not the built-in names. `dir()` does not list the names of built-in functions and variables; if you want a list of those, they are defined in the standard module `builtins` (seen above).

#### Exercise 4.2.2

1. Add a function `fib3(n)` to `fibo.py` that returns the Fibonacci numbers strictly less than `n` **as a generator** (using `yield`) instead of building a list.
2. Reload the module and test your function.

In [ ]:
# Your solution here
generator_addition = '''

def fib3(n):
    a, b = 0, 1
    while a < n:
        yield a
        a, b = b, a + b
'''

with open('fibo.py') as f:
    content = f.read()

# insert before the "if __name__" guard so it stays importable cleanly
marker = 'if __name__ == "__main__":'
content = content.replace(marker, generator_addition.strip('\n') + '\n\n' + marker)

with open('fibo.py', 'w') as f:
    f.write(content)

import importlib
importlib.reload(fibo)
list(fibo.fib3(100))

### 4.2.3 Packages

Packages are a way of structuring Python's module namespace by using "dotted module names". For example, the module name `A.B` designates a submodule named `B` in a package named `A`. Just as the use of modules saves the authors of different modules from having to worry about each other's global variable names, the use of dotted module names saves the authors of multi-module packages from having to worry about each other's module names.

Suppose you want to design a collection of modules (a "package") for the uniform handling of sound files and sound data. There are many different sound file formats, so you may need to create and maintain a growing collection of modules for conversion between formats. There are also many operations you might want to perform on sound data, so you will be writing modules for those too. A possible structure for your package (expressed in terms of a hierarchical filesystem) might look like this:

```
sound/                          Top-level package
      __init__.py                Initialize the sound package
      formats/                   Subpackage for file format conversions
              __init__.py
              wavread.py
              wavwrite.py
              aiffread.py
              aiffwrite.py
              ...
      effects/                   Subpackage for sound effects
              __init__.py
              echo.py
              surround.py
              reverse.py
              ...
      filters/                   Subpackage for filters
              __init__.py
              equalizer.py
              vocoder.py
              karaoke.py
              ...
```

The `__init__.py` files are required to make Python treat directories containing the file as (regular) packages, preventing directories with a common name, such as `string`, from unintentionally hiding valid modules that occur later on the module search path. In the simplest case, `__init__.py` can just be an empty file, but it can also execute initialization code for the package or set the `__all__` variable (described later).

Let's build a small toy package on disk and actually import from it.

In [ ]:
import os

# Build the directory tree
os.makedirs('sound/formats', exist_ok=True)
os.makedirs('sound/effects', exist_ok=True)
os.makedirs('sound/filters', exist_ok=True)

files = {
    'sound/__init__.py': "'''The sound package: tools for handling sound files and data.'''\n",
    'sound/formats/__init__.py': "'''Subpackage for file format conversions.'''\n",
    'sound/formats/wavread.py': (
        'def read(path):\n'
        '    print(f"[wavread] pretending to read {path}")\n'
        '    return b"fake-wav-data"\n'
    ),
    'sound/formats/wavwrite.py': (
        'def write(path, data):\n'
        '    print(f"[wavwrite] pretending to write {len(data)} bytes to {path}")\n'
    ),
    'sound/effects/__init__.py': "'''Subpackage for sound effects.'''\n",
    'sound/effects/echo.py': (
        'def echofilter(data, delay=0.1, atten=0.4):\n'
        '    print(f"[echo] applying echo (delay={delay}, atten={atten}) to {data!r}")\n'
        '    return data\n'
    ),
    'sound/effects/surround.py': (
        'def surround(data, channels=5):\n'
        '    print(f"[surround] spreading {data!r} across {channels} channels")\n'
        '    return data\n'
    ),
    'sound/filters/__init__.py': "'''Subpackage for filters.'''\n",
    'sound/filters/equalizer.py': (
        'def equalize(data, preset="flat"):\n'
        '    print(f"[equalizer] applying {preset!r} preset to {data!r}")\n'
        '    return data\n'
    ),
}

for path, content in files.items():
    with open(path, 'w') as f:
        f.write(content)

print("Package tree created:")
for root, dirs, filenames in os.walk('sound'):
    level = root.replace('sound', '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}{os.path.basename(root) or 'sound'}/")
    for fn in filenames:
        print(f"{indent}  {fn}")

#### Importing from packages

Users of the package can import individual modules from the package, for example:

```python
import sound.effects.echo
```

This loads the submodule `sound.effects.echo`. It must be referenced with its full name, e.g. `sound.effects.echo.echofilter(...)`.

An alternative way of importing the submodule is:

```python
from sound.effects import echo
```

This also loads the submodule `echo`, and makes it available *without* the package prefix, so it can be used as `echo.echofilter(...)`.

Yet another variation is to import the desired function or variable directly:

```python
from sound.effects.echo import echofilter
```

This loads the submodule `echo` again, but makes its function `echofilter()` directly available: `echofilter(...)`.

Let's try each style.

In [ ]:
import sound.effects.echo

sound.effects.echo.echofilter(b"raw-audio", delay=0.2, atten=0.5)

In [ ]:
from sound.effects import surround

surround.surround(b"raw-audio", channels=7)

In [ ]:
from sound.filters.equalizer import equalize

equalize(b"raw-audio", preset="bass-boost")

Note that when using `from package import item`, the item can be either a submodule (or subpackage) of the package, or some other name defined in the package, like a function, class or variable. The `import` statement first tests whether the item is defined in the package; if not, it assumes it is a module and attempts to load it. If it fails to find it, an `ImportError` exception is raised.

Contrarily, when using syntax like `import item.subitem.subsubitem`, each item except for the last must be a package; the last item can be a module or a package, but cannot be a class, function or variable defined in the previous item.

#### Importing `*` from a package

The only solution is for the package author to provide an explicit index of the package. The `import` statement uses the following convention: if a package's `__init__.py` code defines a list named `__all__`, it is taken to be the list of module names that should be imported when `from package import *` is encountered.

Let's give the `effects` subpackage an `__all__` list and see the difference.

In [ ]:
with open('sound/effects/__init__.py', 'w') as f:
    f.write(
        "'''Subpackage for sound effects.'''\n"
        '__all__ = ["echo", "surround"]\n'
    )

# Force a clean reimport of everything under sound.effects
import sys
for mod_name in list(sys.modules):
    if mod_name.startswith('sound'):
        del sys.modules[mod_name]

from sound.effects import *

# Only names listed in __all__ (echo, surround) are now bound here
print("echo module available:", 'echo' in dir())
print("surround module available:", 'surround' in dir())
print("reverse module available:", 'reverse' in dir())  # not in __all__, and doesn't even exist on disk

If `__all__` is not defined, `from sound.effects import *` does **not** import all submodules from the package `sound.effects` into the current namespace — it only ensures that the package `sound.effects` has been imported (running any initialization code in `__init__.py`) and then imports whatever names are defined in the package's namespace, including any submodules explicitly loaded by previous `import` statements.

Remember, using `from package import *` is generally considered bad practice in production code, since it can lead to unreadable code and unknown/shadowed names in a program.

#### Intra-package references

When packages are structured into subpackages, you can use *absolute imports* to refer to submodules of sibling packages. For example, if the module `sound.filters.vocoder` needs to use the `echo` module in `sound.effects`, it can use `from sound.effects import echo`.

You can also write *relative imports*, with the `from module import name` form of import statement. These imports use leading dots to indicate the current and parent packages involved in the relative import. From the `sound.effects.surround` module, for example, you might use:

```python
from . import echo          # sibling module in the same subpackage
from .. import formats      # subpackage of the parent package
from ..filters import equalizer  # sibling subpackage of the parent package
```

Note that relative imports are based on the name of the *current module*. Since the name of the main module is always `"__main__"`, modules intended for use as the main module of a Python application must always use absolute imports.

Let's demonstrate a relative import by editing `surround.py` to use `echo` via a relative import, then run it as part of the package (not as a top-level script, since relative imports require the module to be part of a package).

In [ ]:
with open('sound/effects/surround.py', 'w') as f:
    f.write(
        'from . import echo\n'
        '\n'
        'def surround(data, channels=5):\n'
        '    data = echo.echofilter(data, delay=0.05, atten=0.6)\n'
        '    print(f"[surround] spreading {data!r} across {channels} channels")\n'
        '    return data\n'
    )

# Clean re-import
import sys
for mod_name in list(sys.modules):
    if mod_name.startswith('sound'):
        del sys.modules[mod_name]

from sound.effects.surround import surround
surround(b"raw-audio", channels=7)

#### Packages in multiple directories

Packages support one more special attribute, `__path__`. This is initialized to be a list containing the name of the directory holding the package's `__init__.py` before the code in that file is executed. This variable can be modified; doing so affects future searches for modules and subpackages contained in the package. While this feature is not often needed, it can be used to extend the set of modules found in a package.

In [ ]:
import sound
sound.__path__

#### Exercise 4.2.3

1. Add a `filters/karaoke.py` module with a function `remove_vocals(data)` that just prints a message and returns `data`.
2. Add it to `sound/filters/__init__.py`'s `__all__` list (create `__all__` if it doesn't exist).
3. Import everything from `sound.filters` with `from sound.filters import *` and call `karaoke.remove_vocals(...)`.

In [ ]:
# Your solution here
with open('sound/filters/karaoke.py', 'w') as f:
    f.write(
        'def remove_vocals(data):\n'
        '    print(f"[karaoke] removing vocals from {data!r}")\n'
        '    return data\n'
    )

with open('sound/filters/__init__.py', 'w') as f:
    f.write(
        "'''Subpackage for filters.'''\n"
        '__all__ = ["equalizer", "karaoke"]\n'
    )

import sys
for mod_name in list(sys.modules):
    if mod_name.startswith('sound'):
        del sys.modules[mod_name]

from sound.filters import *
karaoke.remove_vocals(b"raw-audio")

#### Cleaning up

Let's remove the files we created on disk during this notebook so it leaves no clutter behind.

In [ ]:
import shutil, os

for f in ['fibo.py']:
    if os.path.exists(f):
        os.remove(f)

if os.path.exists('sound'):
    shutil.rmtree('sound')

if os.path.exists('__pycache__'):
    shutil.rmtree('__pycache__')

print("Cleaned up.")